# Korean ↔ English Neural Machine Translation — Version 2
## Pretrained Multilingual Transformer + Transfer Learning (NLLB-200-distilled-600M)

This notebook is **Version 2** of the Korean–English NMT project. It does not
replace or modify Version 1 (`v1_seq2seq_attention.ipynb`) — it builds a second,
independent system so the two can be fairly compared.

### Objective
Build a **bidirectional** (Korean→English **and** English→Korean) translation
system using **one** pretrained multilingual Transformer
(`facebook/nllb-200-distilled-600M`), adapted to our Korean-English corpus via
**transfer learning / fine-tuning**, rather than training a sequence model from
scratch as Version 1 did.

### What this notebook will do, end to end
1. Recap Version 1's architecture and diagnosed weaknesses (for context, no code changes).
2. Load and rigorously inspect the same Korean-English dataset used in Version 1.
3. Establish a **pretrained NLLB baseline** (zero-shot, before any fine-tuning) in both directions.
4. Prepare the data for **bidirectional** fine-tuning.
5. Fine-tune NLLB on our corpus.
6. Evaluate the fine-tuned checkpoint against the pretrained baseline and against Version 1.
7. Run qualitative error analysis and make an **evidence-based** model selection —
   the fine-tuned model only "wins" if it measurably outperforms the pretrained baseline.

### Scope of this notebook (this run)
Per the approved project plan, this first pass covers **sections 1–8 only**:
Project Overview → Version 1 Recap → Environment Setup → Imports →
Reproducibility → GPU/VRAM Inspection → Dataset Loading → Dataset Inspection.

Preprocessing, tokenization, and fine-tuning are deliberately **not** included yet
— they follow in the next pass, after reviewing the dataset-quality results below.

### Licensing note (important)
`facebook/nllb-200-distilled-600M` is distributed under **CC-BY-NC-4.0** —
**non-commercial use only**. This project is a non-commercial educational/portfolio
project, so that restriction is acceptable here, but it must not be presented as
suitable for unrestricted commercial deployment. This is documented again in the
project README.

## 1. Version 1 Recap

**Architecture:** Korean text → custom regex-based cleaning (preserving Hangul) →
word-level `TextVectorization` (8,000-token Korean vocab / 5,000-token English vocab)
→ Bidirectional GRU Encoder → single-head Bahdanau (additive) attention → GRU
Decoder → English. Trained from scratch on a Korean-English subset of OPUS-100
(~20,000 sentence pairs, with a small bundled fallback corpus if the download fails).

**Diagnosed weaknesses (carried over from the Version 1 analysis):**

| # | Weakness | Why it hurts translation quality |
|---|---|---|
| 1 | Word-level tokenization on an agglutinative language | Korean particles/endings attach to stems, so surface-form vocabulary explodes; unseen inflected forms become `[UNK]` |
| 2 | Small, narrow training set (~20K pairs) | Not enough data for an RNN trained from scratch to generalize beyond memorized patterns |
| 3 | No pretraining | All linguistic knowledge must come from the 20K pairs; nothing is inherited |
| 4 | Single-head attention | Limited capacity to attend to multiple aspects of a sentence at once; degrades faster on longer/complex input |
| 5 | Unidirectional only | KO→EN and EN→KO would require two entirely separate models with Version 1's design |

**What Version 2 changes:** every one of these five points is addressed by using a
pretrained multilingual Transformer with a subword tokenizer, fine-tuned (not trained
from scratch) on the same data, with built-in bidirectional support via
language-token conditioning.

Version 1's files, model, and notebook are **not modified** by this notebook.

## 2. Environment Setup

Installs the libraries this notebook needs beyond Colab's defaults:
- `transformers` — NLLB model + tokenizer, `Seq2SeqTrainer`
- `sentencepiece` — required by NLLB's tokenizer
- `sacrebleu` — standard, reproducible BLEU scoring
- `evaluate` — HuggingFace metrics wrapper, used here for chrF
- `accelerate` — required by recent `transformers` Trainer versions
- `datasets` — reused from Version 1 to download the OPUS-100 Korean-English subset

Colab already provides `torch`, `pandas`, `numpy`, and `matplotlib`, so those are not
reinstalled.

In [ ]:
# transformers is pinned to a safe range: an earlier, unrelated part of this
# project (the Streamlit app) broke when an unpinned "transformers" pulled in
# a new v5 release with breaking API changes. Section 14 onward uses
# Seq2SeqTrainer / Seq2SeqTrainingArguments with the `eval_strategy` argument
# name, which requires >=4.44 and is safest on 4.x generally.
!pip install -q "transformers>=4.44,<5.0" sentencepiece sacrebleu evaluate accelerate datasets


## 3. Imports

In [ ]:
import os
import re
import json
import random
import pathlib
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch

# transformers / evaluation (used starting in the tokenization/fine-tuning sections;
# imported here so version info can be printed and checked now)
import transformers
import evaluate

print(f"Python torch version:        {torch.__version__}")
print(f"transformers version:        {transformers.__version__}")


## 4. Reproducibility

A fixed random seed is set across every library that introduces randomness
(`random`, `numpy`, `torch`, and CUDA if available), so that the dataset split,
any sampling done during inspection, and later training are reproducible.

In [ ]:
SEED = 42

def set_seed(seed: int = SEED) -> None:
    """Set the random seed across all libraries used in this notebook."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(SEED)
print(f"Global random seed set to {SEED}.")


## 5. GPU / VRAM Inspection

NLLB-200-distilled-600M is roughly 15x larger than Version 1's GRU model. Before
doing anything else, confirm what hardware this session actually has, since that
directly determines feasible batch size, precision (fp16/bf16 vs fp32), and realistic
training time later in this notebook.

**Action if no GPU is detected:** go to `Runtime → Change runtime type → Hardware
accelerator` and select a GPU (T4 is sufficient for fine-tuning a 600M-parameter
model with a small batch size + gradient accumulation) before proceeding past this
point in later sections. The baseline evaluation below can technically run on CPU
but will be considerably slower for a 600M-parameter model.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Selected device: {device}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU: {gpu_name}")
    print(f"Total GPU memory: {total_mem_gb:.1f} GB")

    # fp16 is generally safe and roughly halves memory use on T4-class GPUs.
    use_fp16 = True
    print("fp16 mixed precision: recommended and will be used later in this notebook.")
else:
    print("No GPU detected. Go to Runtime -> Change runtime type -> Hardware "
          "accelerator and select a GPU before running the fine-tuning sections "
          "later in this notebook.")
    use_fp16 = False


## 6. Dataset Loading

Version 2 reuses the same Korean-English dataset **source** as Version 1 (same
OPUS-100 `en-ko` subset, same 20,000-pair cap, same bundled fallback corpus), so
that any quality difference measured later comes from the **modeling approach**,
not from different data.

**One deliberate deviation from Version 1, per review:** Version 1's acquisition
cell applied `MAX_SENTENCE_CHARS = 60`, silently dropping any sentence longer than
60 characters *before* inspection ever saw the data. That made the earlier length
analysis circular (a ">200 characters" check is meaningless when nothing over 60
characters was ever loaded). This cell removes that filter entirely. The only
rejection rule at acquisition time is genuinely empty/missing Korean or English
text. All length-based decisions are deferred until after inspection below, and
ultimately until the NLLB subword-token length distribution is available.

In [ ]:
# ---------------------------------------------------------------------------
# Korean-English Parallel Corpus
# ---------------------------------------------------------------------------
# We try to download a subset of the OPUS-100 (en-ko) corpus from the
# Hugging Face `datasets` hub. If that fails for any reason (no internet
# access, hub unavailable, etc.), we fall back to a small, hand-curated
# sample corpus bundled directly below so the rest of the notebook still
# runs end-to-end.
# ---------------------------------------------------------------------------
NUM_EXAMPLES = 20000          # number of sentence pairs to keep (kept small for a quick tutorial run)

path_to_file = pathlib.Path('kor_eng.txt')

# A small, hand-curated fallback corpus (Korean, English), used only if the
# OPUS-100 download fails (e.g. no internet connection in the runtime).
SAMPLE_PAIRS = [('안녕하세요.', 'Hello.'), ('안녕히 가세요.', 'Goodbye.'), ('감사합니다.', 'Thank you.'), ('천만에요.', "You're welcome."), ('죄송합니다.', "I'm sorry."), ('실례합니다.', 'Excuse me.'), ('만나서 반갑습니다.', 'Nice to meet you.'), ('잘 지내세요?', 'How are you?'), ('저는 잘 지내고 있어요.', 'I am doing well.'), ('이름이 뭐예요?', 'What is your name?'), ('제 이름은 민수입니다.', 'My name is Minsu.'), ('몇 살이에요?', 'How old are you?'), ('저는 스무 살입니다.', 'I am twenty years old.'), ('어디에서 오셨어요?', 'Where are you from?'), ('저는 한국에서 왔습니다.', 'I am from Korea.'), ('오늘 날씨가 좋네요.', 'The weather is nice today.'), ('오늘 날씨가 정말 좋네요.', 'The weather is really nice today.'), ('비가 올 것 같아요.', "It looks like it's going to rain."), ('오늘은 너무 덥습니다.', 'It is too hot today.'), ('겨울에는 눈이 많이 옵니다.', 'It snows a lot in winter.'), ('내일은 흐릴 예정입니다.', 'Tomorrow will be cloudy.'), ('바람이 많이 붑니다.', 'The wind is blowing hard.'), ('오늘 아침은 쌀쌀합니다.', 'It is chilly this morning.'), ('봄이 되면 꽃이 핍니다.', 'Flowers bloom when spring comes.'), ('여름에는 비가 자주 옵니다.', 'It rains often in summer.'), ('저는 매일 아침 일곱 시에 일어납니다.', 'I wake up at seven every morning.'), ('저는 학교에 갑니다.', 'I go to school.'), ('그는 회사에서 일합니다.', 'He works at a company.'), ('우리는 집에서 저녁을 먹습니다.', 'We eat dinner at home.'), ('저는 커피를 좋아합니다.', 'I like coffee.'), ('그녀는 책 읽는 것을 좋아합니다.', 'She likes reading books.'), ('저는 매일 운동을 합니다.', 'I exercise every day.'), ('오늘 저녁에 친구를 만날 거예요.', 'I am going to meet a friend this evening.'), ('주말에는 보통 집에서 쉽니다.', 'I usually rest at home on weekends.'), ('저는 아침에 신문을 읽습니다.', 'I read the newspaper in the morning.'), ('이 음식은 정말 맛있습니다.', 'This food is really delicious.'), ('저는 김치를 매우 좋아합니다.', 'I like kimchi very much.'), ('저녁에 무엇을 먹고 싶어요?', 'What do you want to eat for dinner?'), ('이 식당은 유명합니다.', 'This restaurant is famous.'), ('저는 매운 음식을 잘 못 먹습니다.', "I can't eat spicy food well."), ('밥을 먹었어요?', 'Have you eaten?'), ('저는 아직 점심을 안 먹었어요.', "I haven't had lunch yet."), ('이 커피는 너무 답니다.', 'This coffee is too sweet.'), ('그는 요리를 잘합니다.', 'He cooks well.'), ('우리는 함께 저녁을 먹었습니다.', 'We had dinner together.'), ('저는 여행을 좋아합니다.', 'I like traveling.'), ('다음 주에 부산에 갈 거예요.', 'I am going to Busan next week.'), ('공항까지 얼마나 걸려요?', 'How long does it take to the airport?'), ('저는 처음으로 서울에 왔습니다.', 'This is my first time coming to Seoul.'), ('기차표를 예매했어요.', 'I booked a train ticket.'), ('이 지역은 관광지로 유명합니다.', 'This area is famous as a tourist spot.'), ('저는 제주도에 가본 적이 있습니다.', 'I have been to Jeju Island.'), ('호텔은 어디에 있어요?', 'Where is the hotel?'), ('여권을 보여 주세요.', 'Please show me your passport.'), ('우리는 비행기를 타고 여행했습니다.', 'We traveled by airplane.'), ('저는 인공지능을 공부하고 있습니다.', 'I am studying artificial intelligence.'), ('그는 대학교에서 컴퓨터 공학을 전공했습니다.', 'He majored in computer engineering at university.'), ('저는 회의에 참석해야 합니다.', 'I have to attend a meeting.'), ('이 프로젝트는 다음 달에 끝납니다.', 'This project ends next month.'), ('저는 매일 컴퓨터로 일합니다.', 'I work with a computer every day.'), ('그녀는 열심히 공부합니다.', 'She studies hard.'), ('저는 새로운 언어를 배우고 있습니다.', 'I am learning a new language.'), ('우리 팀은 새로운 프로그램을 개발하고 있습니다.', 'Our team is developing a new program.'), ('저는 오늘 시험이 있습니다.', 'I have an exam today.'), ('그는 대학원에서 인공지능을 연구합니다.', 'He researches artificial intelligence in graduate school.'), ('저는 세 형제가 있습니다.', 'I have three siblings.'), ('제 아버지는 의사입니다.', 'My father is a doctor.'), ('제 어머니는 선생님입니다.', 'My mother is a teacher.'), ('저는 동생이 한 명 있습니다.', 'I have one younger sibling.'), ('우리 가족은 함께 여행을 갑니다.', 'Our family goes on trips together.'), ('그는 제 친한 친구입니다.', 'He is my close friend.'), ('저는 할머니를 자주 찾아뵙니다.', 'I often visit my grandmother.'), ('그녀는 저의 동료입니다.', 'She is my colleague.'), ('저는 오늘 부모님을 뵈러 갑니다.', 'I am going to see my parents today.'), ('우리는 오랜 친구입니다.', 'We are old friends.'), ('이 스마트폰은 아주 빠릅니다.', 'This smartphone is very fast.'), ('인공지능 기술이 빠르게 발전하고 있습니다.', 'Artificial intelligence technology is developing rapidly.'), ('저는 매일 인터넷을 사용합니다.', 'I use the internet every day.'), ('이 프로그램은 사용하기 쉽습니다.', 'This program is easy to use.'), ('새로운 노트북을 샀습니다.', 'I bought a new laptop.'), ('이 앱은 매우 유용합니다.', 'This app is very useful.'), ('그는 소프트웨어 개발자입니다.', 'He is a software developer.'), ('데이터는 클라우드에 저장됩니다.', 'The data is stored in the cloud.'), ('저는 매일 이메일을 확인합니다.', 'I check my email every day.'), ('인공지능은 우리의 삶을 바꾸고 있습니다.', 'Artificial intelligence is changing our lives.'), ('저는 오늘 기분이 좋습니다.', 'I feel good today.'), ('그는 매우 행복해 보입니다.', 'He looks very happy.'), ('저는 조금 피곤합니다.', 'I am a little tired.'), ('그녀는 걱정이 많습니다.', 'She worries a lot.'), ('저는 이 영화가 정말 재미있었습니다.', 'I found this movie really fun.'), ('그것은 좋은 생각인 것 같습니다.', 'That seems like a good idea.'), ('저는 그 소식을 듣고 놀랐습니다.', 'I was surprised to hear the news.'), ('저는 당신의 의견에 동의합니다.', 'I agree with your opinion.'), ('그는 항상 긍정적입니다.', 'He is always positive.'), ('저는 오늘 하루가 정말 즐거웠습니다.', 'I really enjoyed today.'), ('이것은 얼마예요?', 'How much is this?'), ('화장실이 어디에 있어요?', 'Where is the restroom?'), ('조금만 기다려 주세요.', 'Please wait a moment.'), ('문을 좀 열어 주시겠어요?', 'Could you please open the door?'), ('다시 한 번 말씀해 주시겠어요?', 'Could you say that again?'), ('여기에 앉아도 될까요?', 'May I sit here?'), ('창문을 닫아 주세요.', 'Please close the window.'), ('도와주셔서 감사합니다.', 'Thank you for helping me.'), ('천천히 말씀해 주세요.', 'Please speak slowly.'), ('이 근처에 은행이 있어요?', 'Is there a bank near here?'), ('시간이 얼마나 걸려요?', 'How long does it take?'), ('저는 음악 듣는 것을 좋아합니다.', 'I like listening to music.'), ('그 영화는 아직 안 봤어요.', "I haven't seen that movie yet."), ('오늘은 제 생일입니다.', 'Today is my birthday.'), ('우리는 내일 만날 예정입니다.', 'We plan to meet tomorrow.'), ('이 책은 매우 유익합니다.', 'This book is very informative.'), ('저는 운전하는 것을 좋아하지 않습니다.', "I don't like driving."), ('그는 매일 아침 커피를 마십니다.', 'He drinks coffee every morning.'), ('오늘 회의는 취소되었습니다.', "Today's meeting was canceled."), ('저는 그 문제를 아직 해결하지 못했습니다.', "I haven't solved that problem yet."), ('이 도시는 매우 아름답습니다.', 'This city is very beautiful.'), ('저는 새로운 취미를 시작했습니다.', 'I started a new hobby.'), ('우리는 함께 영화를 보러 갔습니다.', 'We went to watch a movie together.'), ('그는 자주 늦게 일어납니다.', 'He often wakes up late.'), ('저는 매일 일기를 씁니다.', 'I write a diary every day.'), ('이 옷은 저에게 너무 큽니다.', 'This clothing is too big for me.'), ('그녀는 노래를 매우 잘 부릅니다.', 'She sings very well.'), ('저는 이번 주말에 이사합니다.', 'I am moving this weekend.'), ('우리는 새로운 계획을 세웠습니다.', 'We made a new plan.'), ('그는 회의에 늦었습니다.', 'He was late for the meeting.'), ('저는 아직 그 소설을 다 읽지 못했습니다.', "I haven't finished reading that novel yet."), ('이 길은 매우 복잡합니다.', 'This road is very busy.'), ('저는 매일 일곱 시간 동안 잡니다.', 'I sleep for seven hours every day.'), ('그녀는 세 개의 언어를 할 수 있습니다.', 'She can speak three languages.'), ('저는 오늘 중요한 발표가 있습니다.', 'I have an important presentation today.'), ('이 문제는 정말 어렵습니다.', 'This problem is really difficult.'), ('저는 어제 그를 만났습니다.', 'I met him yesterday.'), ('그들은 내년에 결혼할 예정입니다.', 'They plan to get married next year.'), ('저는 이 회사에서 오 년 동안 일했습니다.', 'I have worked at this company for five years.'), ('그는 매일 신문을 읽습니다.', 'He reads the newspaper every day.'), ('저는 오늘 아침에 늦게 일어났습니다.', 'I woke up late this morning.'), ('우리는 지난주에 산에 올라갔습니다.', 'We climbed the mountain last week.'), ('이 상품은 매우 인기가 많습니다.', 'This product is very popular.'), ('저는 아직 그 이메일에 답장하지 못했습니다.', "I haven't replied to that email yet."), ('그녀는 매우 친절합니다.', 'She is very kind.')]


def download_opus100(num_examples=NUM_EXAMPLES):
    """Download a subset of OPUS-100 (en-ko) via the Hugging Face `datasets` library.

    Unlike Version 1, this does NOT filter by sentence length -- only empty or
    missing Korean/English strings are rejected here. Length-based filtering
    (if any) is a preprocessing decision made later, after inspecting both the
    raw character-length distribution (below) and the NLLB subword-token
    length distribution (in the tokenization section).
    """
    from datasets import load_dataset

    print("Downloading OPUS-100 (en-ko) from the Hugging Face Hub ...")
    ds = load_dataset("Helsinki-NLP/opus-100", "en-ko", split="train")

    pairs = []
    for ex in ds:
        ko = ex["translation"]["ko"].strip()
        en = ex["translation"]["en"].strip()
        if not ko or not en:
            continue
        pairs.append((ko, en))
        if len(pairs) >= num_examples:
            break
    return pairs


try:
    pairs = download_opus100()
    if len(pairs) < 500:
        raise RuntimeError("Downloaded dataset was too small.")
    dataset_source = "OPUS-100 (Helsinki-NLP, via Hugging Face Datasets)"
except Exception as e:
    print(f"Falling back to the bundled sample corpus. Reason: {e}")
    pairs = SAMPLE_PAIRS
    dataset_source = "bundled sample corpus"

lines = [f"{ko}\t{en}" for ko, en in pairs]
path_to_file.write_text("\n".join(lines), encoding="utf-8")

print(f"Dataset source : {dataset_source}")
print(f"Sentence pairs : {len(pairs)}")
print(f"Saved to       : {path_to_file.resolve()}")


In [ ]:
# Load the pairs into a DataFrame for the inspection work in the next section.
# `pairs` and `dataset_source` come from the cell above.
df = pd.DataFrame(pairs, columns=["korean", "english"])
print(f"Loaded {len(df)} sentence pairs from: {dataset_source}")
df.head()


## 7. Dataset Inspection and Quality Analysis

Before touching tokenization or training, we inspect the **unfiltered** data:
size, duplicates, missing values, sentence-length distribution in both directions
(character-, word/eojeol-, and percentile-level), obvious corruption/HTML
artifacts (with a categorized breakdown, not just a raw count), and a manual
spot-check of Korean/English alignment. Cleaning happens in the *next* notebook
section (preprocessing) — this section only measures and reports.

**Note on duplicate terminology:** the checks below detect **exact pair
duplicates** (identical Korean *and* identical English) and **exact source/target
duplicates** (identical Korean text regardless of English, and vice versa). These
are exact-string checks, not fuzzy/near-duplicate similarity matching — we are
not running any approximate-matching algorithm here, deliberately, to keep this
inspection lightweight.

In [ ]:
# --- Basic size and missing-value checks -----------------------------------
print(f"Total pairs loaded:        {len(df)}")
print(f"Missing Korean values:     {df['korean'].isna().sum()}")
print(f"Missing English values:    {df['english'].isna().sum()}")
print(f"Empty Korean strings:      {(df['korean'].str.strip() == '').sum()}")
print(f"Empty English strings:     {(df['english'].str.strip() == '').sum()}")


In [ ]:
# --- Exact / source / target duplicate checks --------------------------------
# These are exact-string checks (not fuzzy/near-duplicate similarity):
#   - "exact pair" duplicates: identical Korean AND identical English
#   - "duplicate Korean source" : same Korean text, any English (may be a
#     legitimate one-to-many translation, e.g. multiple valid English phrasings)
#   - "duplicate English target": same English text, any Korean (same caveat,
#     in the other direction)
# Handling plan (applied later, in preprocessing -- not here):
#   - Exact pair duplicates WILL be removed before the train/validation/test
#     split, to reduce leakage risk (an identical pair could otherwise land in
#     both train and test).
#   - Duplicate Korean-only or English-only rows will NOT be blanket-removed,
#     since a single source sentence can have more than one valid translation.
exact_dupe_mask = df.duplicated(subset=["korean", "english"])
ko_dupe_mask = df.duplicated(subset=["korean"])
en_dupe_mask = df.duplicated(subset=["english"])

print(f"Exact duplicate pairs (same KO and same EN):  {exact_dupe_mask.sum()}")
print(f"Duplicate Korean source sentences (any EN):   {ko_dupe_mask.sum()}")
print(f"Duplicate English target sentences (any KO):  {en_dupe_mask.sum()}")

if exact_dupe_mask.sum() > 0:
    print("\nSample exact duplicate pairs:")
    print(df[df.duplicated(subset=["korean", "english"], keep=False)][["korean", "english"]].head(6))

if ko_dupe_mask.sum() > 0:
    print("\nSample repeated Korean source with differing English (legitimate one-to-many candidates):")
    repeated_ko = df[df.duplicated(subset=["korean"], keep=False)].sort_values("korean")
    print(repeated_ko[["korean", "english"]].head(6))


In [ ]:
# --- Sentence length distribution (characters and whitespace-split words) ---
df["ko_char_len"] = df["korean"].str.len()
df["en_char_len"] = df["english"].str.len()
df["ko_word_len"] = df["korean"].str.split().str.len()
df["en_word_len"] = df["english"].str.split().str.len()

length_cols = ["ko_char_len", "en_char_len", "ko_word_len", "en_word_len"]

# Percentiles matter more than mean/std here: NLLB's max_source_length /
# max_target_length choice later should be driven by where the bulk of the
# distribution sits (e.g. p95/p99), not by the mean or the single max outlier.
percentiles = [0.50, 0.75, 0.90, 0.95, 0.99]
length_stats = df[length_cols].describe(percentiles=percentiles)
length_stats


**Reminder:** the character-length statistics and percentiles above (and the
threshold counts below) are exploratory only. The final `max_source_length` /
`max_target_length` for NLLB fine-tuning will be chosen from the tokenizer's
**subword-token** length distribution, in the tokenization section — not from
raw character counts, since NLLB's SentencePiece tokenizer does not map 1
character to 1 token, especially for Korean.

In [ ]:
# --- Counts above useful character-length thresholds --------------------------
thresholds = [60, 100, 150, 200]

print("Pairs where at least one side exceeds the threshold (character count):\n")
for t in thresholds:
    count = ((df["ko_char_len"] > t) | (df["en_char_len"] > t)).sum()
    pct = 100 * count / len(df)
    print(f"  > {t:>3} chars: {count:>6} pairs ({pct:.2f}% of the dataset)")

print("\nPer-language breakdown:\n")
for t in thresholds:
    ko_count = (df["ko_char_len"] > t).sum()
    en_count = (df["en_char_len"] > t).sum()
    print(f"  > {t:>3} chars -- Korean: {ko_count:>6}   English: {en_count:>6}")


In [ ]:
# --- Very short / very long sentence flags ----------------------------------
# Thresholds are intentionally generous at this stage -- this is measurement,
# not filtering. Actual filtering decisions are made in the preprocessing section
# based on what these numbers show.
MIN_CHAR_LEN = 2
MAX_CHAR_LEN = 200

too_short = df[(df["ko_char_len"] < MIN_CHAR_LEN) | (df["en_char_len"] < MIN_CHAR_LEN)]
too_long = df[(df["ko_char_len"] > MAX_CHAR_LEN) | (df["en_char_len"] > MAX_CHAR_LEN)]

print(f"Pairs with a side shorter than {MIN_CHAR_LEN} characters: {len(too_short)}")
print(f"Pairs with a side longer than {MAX_CHAR_LEN} characters:  {len(too_long)}")

if len(too_short) > 0:
    print("\nSample very-short pairs:")
    print(too_short[["korean", "english"]].head(5))

if len(too_long) > 0:
    print("\nSample very-long pairs:")
    print(too_long[["korean", "english"]].head(3))


In [ ]:
# --- Length distribution plots ----------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df["ko_char_len"], bins=40, alpha=0.7, label="Korean")
axes[0].hist(df["en_char_len"], bins=40, alpha=0.7, label="English")
axes[0].set_title("Sentence length (characters)")
axes[0].set_xlabel("Characters")
axes[0].set_ylabel("Count")
axes[0].legend()

axes[1].hist(df["ko_word_len"], bins=30, alpha=0.7, label="Korean (whitespace units)")
axes[1].hist(df["en_word_len"], bins=30, alpha=0.7, label="English (words)")
axes[1].set_title("Sentence length (word/eojeol count)")
axes[1].set_xlabel("Word / eojeol count")
axes[1].set_ylabel("Count")
axes[1].legend()

plt.tight_layout()
plt.show()


### Categorizing unexpected characters (not just counting them)

429 Korean rows were flagged by a simple "not Hangul/Latin/digit/basic-punctuation"
regex in the previous pass. A raw count like that conflates very different
situations: a stray HTML entity is a corruption problem; a sentence that legitimately
mixes in a Hanja (Chinese character) name or an em-dash is not. The cell below
categorizes every flagged character by Unicode block/category and shows
representative example sentences per category, so we can decide — in the
preprocessing section, not here — which categories are genuine corruption versus
valid mixed-language content. Nothing is removed in this cell.

In [ ]:
import unicodedata

# Baseline "expected" character set: whitespace, Hangul syllables/Jamo, digits,
# basic Latin letters, and a small set of common punctuation.
expected_char_pattern = re.compile(r"[\sㄱ-ㅎㅏ-ㅣ가-힣0-9a-zA-Z.,!?\'\"\-:;()%]")

def categorize_char(ch: str) -> str:
    """Bucket a single unexpected character into a human-readable category."""
    code_point = ord(ch)
    if 0x4E00 <= code_point <= 0x9FFF:
        return "CJK/Hanja"
    if 0x3040 <= code_point <= 0x30FF:
        return "Japanese Hiragana/Katakana"
    if unicodedata.category(ch).startswith("P"):
        return "Unicode punctuation (non-ASCII, e.g. curly quotes, em-dash)"
    if unicodedata.category(ch).startswith("S"):
        return "Symbol (currency, math, misc.)"
    if 0x00C0 <= code_point <= 0x024F:
        return "Accented Latin (e.g. French/Spanish/German letters)"
    if code_point < 0x20 or code_point == 0x7F:
        return "Control character"
    return f"Other (Unicode category {unicodedata.category(ch)})"

def find_unexpected_chars(text: str):
    return [ch for ch in text if not expected_char_pattern.match(ch)]

df["ko_unexpected_chars"] = df["korean"].apply(find_unexpected_chars)
flagged_rows = df[df["ko_unexpected_chars"].apply(len) > 0]

print(f"Korean rows with at least one unexpected character: {len(flagged_rows)}")

category_counter = Counter()
category_examples = {}
for _, row in flagged_rows.iterrows():
    for ch in row["ko_unexpected_chars"]:
        category = categorize_char(ch)
        category_counter[category] += 1
        category_examples.setdefault(category, set()).add(row["korean"])

print("\nBreakdown by category (character-level counts):\n")
for category, count in category_counter.most_common():
    print(f"  {category:<55} {count}")


In [ ]:
# --- Representative example sentences per category ---------------------------
print("Representative example sentences per category (up to 3 each):\n")
for category in category_counter:
    examples = list(category_examples[category])[:3]
    print(f"[{category}]")
    for ex in examples:
        print(f"  {ex}")
    print()


In [ ]:
# --- Lightweight English-side corruption check (HTML tags / control chars) ---
# The Korean side gets the detailed categorization above because Korean is the
# language most at risk of subtle corruption in this pipeline; English gets a
# simpler pass here since it's a low-risk, high-resource language for this check.
html_pattern = re.compile(r"<[^>]+>|&[a-zA-Z]+;")
control_char_pattern = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")

en_html_hits = df["english"].str.contains(html_pattern, na=False).sum()
en_control_hits = df["english"].str.contains(control_char_pattern, na=False).sum()
print(f"[English] HTML-like tags/entities: {en_html_hits}")
print(f"[English] Control characters:      {en_control_hits}")

if en_html_hits > 0:
    print("\nSample English rows with HTML-like content:")
    print(df[df["english"].str.contains(html_pattern, na=False)]["english"].head(5).tolist())


In [ ]:
# --- Manual alignment spot-check ---------------------------------------------
# Randomly sample a handful of pairs to eyeball Korean/English alignment.
# This is a human-in-the-loop sanity check, not an automated metric.
set_seed(SEED)
sample_idx = np.random.choice(len(df), size=8, replace=False)

print("Random sample for manual Korean/English alignment review:\n")
for i in sample_idx:
    row = df.iloc[i]
    print(f"KO: {row['korean']}")
    print(f"EN: {row['english']}")
    print("-" * 60)


### Summary of this section

Run the cells above (on the now-unfiltered 20K sample) and review:
- total pair count, missing/empty values
- exact pair duplicates, and duplicate Korean-source / English-target counts
  (exact-string checks, not fuzzy similarity — see the handling-plan note above)
- character-length percentiles (p50/p75/p90/p95/p99/max) and word/eojeol-count
  statistics, for both languages
- counts of pairs exceeding 60/100/150/200 characters, both combined and
  per-language
- the length-distribution plots
- the categorized unexpected-character breakdown (CJK/Hanja, Unicode punctuation,
  symbols, accented Latin, control characters, etc.) with representative examples
- the English-side HTML/control-character check
- the manual alignment spot-check output

**Stopping point (Sections 1–8):** per the project plan, this was the first
review checkpoint. Sections 9 onward (below) implement cleaning, deduplication,
the leakage-safe split, tokenizer loading, subword-length analysis, and the
pretrained baseline — fine-tuning itself is deliberately not included yet and
follows in a later pass.

## 8. Conservative Data Cleaning

Version 1's preprocessing applied a single aggressive regex whitelist to Korean
text. We deliberately do **not** repeat that here. Instead, cleaning is a short
sequence of narrow, individually-audited rules, each applied to a working copy
of the data (`df_clean`), with an explicit **before/after count** printed after
every rule so the effect of each step is visible on its own.

**What is preserved on purpose:** legitimate Korean punctuation, Hanja
(Chinese characters that can legitimately appear in Korean text, e.g. in names
or formal writing), Latin names, numbers, URLs, and ordinary mixed-language
content. None of the rules below strip characters based on a whitelist — they
only target specific, identifiable noise patterns found during inspection
(HTML residue, control characters, irregular whitespace, and rows with no
real content left after cleaning).

**Rules applied, in order:**
1. Unicode NFKC normalization (both languages)
2. HTML entity/residue cleanup (e.g. `&nbsp;`, literal leftover `nbsp;`, `&amp;`, etc.)
3. Control-character removal
4. Whitespace normalization (collapse runs of whitespace, strip ends)
5. Drop rows where a side has **no real content** left (no Hangul, Latin letter,
   or digit at all) after the rules above — this is the only row-removal rule
   in this section, and it only removes genuinely unusable rows, not merely
   unusual ones.

In [ ]:
import unicodedata as _ud
import html as html_lib

df_clean = df[["korean", "english"]].copy()
print(f"Starting rows: {len(df_clean)}")


def nfkc_normalize(text: str) -> str:
    return _ud.normalize("NFKC", text)


def clean_html_residue(text: str) -> str:
    # Decode standard named/numeric HTML entities (&nbsp; &amp; &lt; &quot; ...).
    text = html_lib.unescape(text)
    # Handle "leftover" entity fragments missing their leading "&" -- these
    # appear in some scraped corpora as literal text (e.g. "nbsp;") after an
    # earlier, imperfect decoding step upstream of OPUS-100.
    text = re.sub(r"\b(nbsp|amp|lt|gt|quot|apos);", " ", text)
    return text


_control_char_pattern = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")

def remove_control_chars(text: str) -> str:
    return _control_char_pattern.sub("", text)


def normalize_whitespace(text: str) -> str:
    text = text.replace("\xa0", " ")  # non-breaking space -> regular space
    text = re.sub(r"\s+", " ", text).strip()
    return text


def has_real_content(text: str) -> bool:
    """True if the text contains at least one Unicode letter or number,
    regardless of script (Hangul, Hanja/CJK, Latin, accented Latin, etc.).
    Uses unicodedata.category() rather than a script-specific whitelist, so a
    legitimate sentence in any script is never dropped just for using
    characters outside Hangul/ASCII Latin. Punctuation-only / symbol-only /
    empty strings still return False."""
    return any(_ud.category(ch)[0] in ("L", "N") for ch in text)


In [ ]:
# --- Rule 1: Unicode NFKC normalization --------------------------------------
ko_before = df_clean["korean"].copy()
en_before = df_clean["english"].copy()

df_clean["korean"] = df_clean["korean"].apply(nfkc_normalize)
df_clean["english"] = df_clean["english"].apply(nfkc_normalize)

ko_changed = (df_clean["korean"] != ko_before).sum()
en_changed = (df_clean["english"] != en_before).sum()
print(f"Rule 1 (NFKC normalization): {ko_changed} Korean rows modified, "
      f"{en_changed} English rows modified. Row count: {len(df_clean)} (unchanged).")


In [ ]:
# --- Rule 2: HTML entity/residue cleanup -------------------------------------
ko_before = df_clean["korean"].copy()
en_before = df_clean["english"].copy()

df_clean["korean"] = df_clean["korean"].apply(clean_html_residue)
df_clean["english"] = df_clean["english"].apply(clean_html_residue)

ko_changed = (df_clean["korean"] != ko_before).sum()
en_changed = (df_clean["english"] != en_before).sum()
print(f"Rule 2 (HTML entity/residue cleanup): {ko_changed} Korean rows modified, "
      f"{en_changed} English rows modified. Row count: {len(df_clean)} (unchanged).")

if en_changed > 0:
    print("\nSample English rows changed by this rule:")
    changed_idx = en_before[en_before != df_clean["english"]].index[:5]
    for i in changed_idx:
        print(f"  before: {en_before[i]!r}")
        print(f"  after:  {df_clean.loc[i, 'english']!r}")


In [ ]:
# --- Rule 3: control-character removal ---------------------------------------
ko_before = df_clean["korean"].copy()
en_before = df_clean["english"].copy()

df_clean["korean"] = df_clean["korean"].apply(remove_control_chars)
df_clean["english"] = df_clean["english"].apply(remove_control_chars)

ko_changed = (df_clean["korean"] != ko_before).sum()
en_changed = (df_clean["english"] != en_before).sum()
print(f"Rule 3 (control-character removal): {ko_changed} Korean rows modified, "
      f"{en_changed} English rows modified. Row count: {len(df_clean)} (unchanged).")


In [ ]:
# --- Rule 4: whitespace normalization -----------------------------------------
ko_before = df_clean["korean"].copy()
en_before = df_clean["english"].copy()

df_clean["korean"] = df_clean["korean"].apply(normalize_whitespace)
df_clean["english"] = df_clean["english"].apply(normalize_whitespace)

ko_changed = (df_clean["korean"] != ko_before).sum()
en_changed = (df_clean["english"] != en_before).sum()
print(f"Rule 4 (whitespace normalization): {ko_changed} Korean rows modified, "
      f"{en_changed} English rows modified. Row count: {len(df_clean)} (unchanged).")


In [ ]:
# --- Rule 5: drop rows with no real content on either side --------------------
before_n = len(df_clean)
no_content_mask = ~(df_clean["korean"].apply(has_real_content) &
                     df_clean["english"].apply(has_real_content))

print(f"Rows with no real content on at least one side: {no_content_mask.sum()}")
if no_content_mask.sum() > 0:
    print("\nSample no-content rows (about to be dropped):")
    print(df_clean[no_content_mask][["korean", "english"]].head(10))

df_clean = df_clean[~no_content_mask].reset_index(drop=True)
print(f"\nRule 5 (drop no-content rows): {before_n} -> {len(df_clean)} "
      f"({before_n - len(df_clean)} removed)")


### Reporting-only check: code/metadata-like strings

Manual inspection also flagged some rows that look like code or metadata
fragments (e.g. bracketed placeholders, repeated underscores, template-like
tokens) rather than natural sentences. This check **reports** candidates for
your review — it does not remove anything, since some of these may still be
legitimate (e.g. a sentence that genuinely quotes a filename or a tag).

In [ ]:
# --- Reporting-only: possible code/metadata-like rows -------------------------
code_like_pattern = re.compile(r"[{}\[\]<>]|_{2,}|\b[A-Z0-9_]{6,}\b")

code_like_mask = (df_clean["korean"].str.contains(code_like_pattern, na=False) |
                  df_clean["english"].str.contains(code_like_pattern, na=False))

print(f"Rows flagged as possibly code/metadata-like: {code_like_mask.sum()} "
      f"(NOT removed -- for manual review only)")
if code_like_mask.sum() > 0:
    print("\nSample flagged rows:")
    print(df_clean[code_like_mask][["korean", "english"]].head(10))


### Cleaning summary

In [ ]:
print("=" * 60)
print("Cleaning summary")
print("=" * 60)
print(f"Rows before cleaning:  {len(df)}")
print(f"Rows after cleaning:   {len(df_clean)}")
print(f"Rows removed:          {len(df) - len(df_clean)} "
      f"({100 * (len(df) - len(df_clean)) / len(df):.2f}%)")
print("(Text-normalization rules 1-4 modify content without removing rows; "
      "only Rule 5 removes rows.)")


## 9. Exact Deduplication

Exact pair duplicates are removed **now, after cleaning** (not before) --
cleaning can itself create *new* exact duplicates that did not exist in the raw
data (e.g. two originally-different strings that become identical once HTML
residue and whitespace are normalized). Per the project decision, only exact
`(korean, english)` pair duplicates are removed. Rows that merely share a
Korean source or an English target with another row are **kept**, since a
single sentence can have more than one legitimate translation.

In [ ]:
before_n = len(df_clean)
exact_dupe_mask = df_clean.duplicated(subset=["korean", "english"], keep="first")
print(f"Exact duplicate pairs found after cleaning: {exact_dupe_mask.sum()}")

df_clean = df_clean[~exact_dupe_mask].reset_index(drop=True)
print(f"Exact deduplication: {before_n} -> {len(df_clean)} "
      f"({before_n - len(df_clean)} removed)")

# Re-check remaining source/target duplicate counts for transparency -- these
# are intentionally NOT removed, just reported.
remaining_ko_dupes = df_clean.duplicated(subset=["korean"]).sum()
remaining_en_dupes = df_clean.duplicated(subset=["english"]).sum()
print(f"\nRemaining (kept) duplicate Korean sources:  {remaining_ko_dupes}")
print(f"Remaining (kept) duplicate English targets: {remaining_en_dupes}")


## 10. Leakage-Safe Train / Validation / Test Split

A plain random row-level split would risk leakage: since some Korean sources
and English targets repeat across rows (kept deliberately in Section 9), a
naive random split could place the same Korean sentence in both the training
set and the test set, letting the model "cheat" by having memorized that exact
source during training.

**Strategy — connected-component grouping:** build a graph where each row is a
node, and two rows are connected by an edge if they share the same Korean
source **or** the same English target. Every connected component (a group of
rows transitively linked by shared sources/targets) is then assigned **as a
whole** to train, validation, or test — never split across sets. This
guarantees zero Korean-source and zero English-target overlap between splits
by construction, which a random row-level split cannot guarantee.

Components are assigned using a greedy heuristic: process components
largest-first, and each time assign the component to whichever split is
currently furthest below its target proportion (90% / 5% / 5% by default).
This keeps the realized split sizes close to the targets despite components
having different sizes. The random seed fixed in Section 4 makes this
reproducible.

In [ ]:
# --- Union-Find (disjoint-set) to group rows sharing a Korean source or an
# English target into connected components ------------------------------------
class DisjointSet:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]  # path compression
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1


n_rows = len(df_clean)
dsu = DisjointSet(n_rows)

ko_groups = df_clean.groupby("korean").indices   # value -> array of row indices
en_groups = df_clean.groupby("english").indices

for indices in ko_groups.values():
    for i in range(1, len(indices)):
        dsu.union(indices[0], indices[i])

for indices in en_groups.values():
    for i in range(1, len(indices)):
        dsu.union(indices[0], indices[i])

component_of = np.array([dsu.find(i) for i in range(n_rows)])
n_components = len(set(component_of.tolist()))
print(f"Rows: {n_rows}")
print(f"Connected components (leakage-safe groups): {n_components}")
print(f"Rows that are their own singleton component: "
      f"{sum(1 for c in Counter(component_of.tolist()).values() if c == 1)}")


### Connected-component size diagnostics (before assigning splits)

A component can grow large if many rows share a generic, frequently-repeated
Korean or English string (e.g. "예." / "Okay." / "Yes."). A single very large
component would force an entire cluster of otherwise-unrelated rows into one
split together, which can distort the intended 90/5/5 proportions. This cell
reports component-size statistics **before** any split assignment happens, so
that possibility is visible up front rather than discovered after the fact.

In [ ]:
# --- Connected-component size diagnostics -------------------------------------
component_sizes = Counter(component_of.tolist())
sizes_array = np.array(list(component_sizes.values()))
components_sorted = sorted(component_sizes.items(), key=lambda kv: (-kv[1], kv[0]))

n_singletons = int((sizes_array == 1).sum())

print(f"Total connected components:     {len(component_sizes)}")
print(f"Singleton components (size 1):  {n_singletons} "
      f"({100 * n_singletons / len(component_sizes):.2f}% of components)")
print(f"Largest component size:         {sizes_array.max()}")
print()
print("Component-size percentiles:")
for p in [50, 90, 95, 99]:
    print(f"  p{p}: {np.percentile(sizes_array, p):.1f}")
print(f"  max: {sizes_array.max()}")

print("\nTop 10 largest components (id, size, and a sample row from each):")
for component_id, size in components_sorted[:10]:
    sample_row_idx = np.where(component_of == component_id)[0][0]
    sample_ko = df_clean.iloc[sample_row_idx]["korean"]
    sample_en = df_clean.iloc[sample_row_idx]["english"]
    print(f"  component {component_id:>6}  size={size:>5}   "
          f"sample: KO={sample_ko!r}  EN={sample_en!r}")

largest_component_share = sizes_array.max() / n_rows
print(f"\nLargest component as a share of the full dataset: "
      f"{100 * largest_component_share:.2f}%")
if largest_component_share > 0.05:
    print("NOTE: the largest component exceeds 5% of the dataset on its own -- "
          "review the sample above; if it is a generic repeated phrase, consider "
          "whether it belongs in this leakage-safe grouping at all before "
          "proceeding to the split assignment below.")


In [ ]:
# --- Greedy component-to-split assignment -------------------------------------
# Reuses `component_sizes` and `components_sorted` computed in the diagnostics
# cell above.
set_seed(SEED)

TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.90, 0.05, 0.05
assert abs(TRAIN_FRAC + VAL_FRAC + TEST_FRAC - 1.0) < 1e-9

target_frac = {"train": TRAIN_FRAC, "val": VAL_FRAC, "test": TEST_FRAC}
assigned_count = {"train": 0, "val": 0, "test": 0}
component_to_split = {}

for component_id, size in components_sorted:
    total_assigned = sum(assigned_count.values())
    if total_assigned == 0:
        deficits = target_frac
    else:
        deficits = {
            split: target_frac[split] - (assigned_count[split] / total_assigned)
            for split in target_frac
        }
    chosen_split = max(deficits, key=deficits.get)
    component_to_split[component_id] = chosen_split
    assigned_count[chosen_split] += size

split_labels = np.array([component_to_split[c] for c in component_of])
df_clean["split"] = split_labels

for split in ["train", "val", "test"]:
    n = (df_clean["split"] == split).sum()
    print(f"{split:>5}: {n:>6} rows ({100 * n / len(df_clean):.2f}%)")


### Post-split overlap verification (should be zero for source/target overlap)

In [ ]:
train_df = df_clean[df_clean["split"] == "train"].reset_index(drop=True)
val_df = df_clean[df_clean["split"] == "val"].reset_index(drop=True)
test_df = df_clean[df_clean["split"] == "test"].reset_index(drop=True)

print(f"Train: {len(train_df)}   Val: {len(val_df)}   Test: {len(test_df)}")

def overlap_report(a_df, b_df, name_a, name_b):
    exact_a = set(zip(a_df["korean"], a_df["english"]))
    exact_b = set(zip(b_df["korean"], b_df["english"]))
    ko_a, ko_b = set(a_df["korean"]), set(b_df["korean"])
    en_a, en_b = set(a_df["english"]), set(b_df["english"])

    print(f"\n{name_a} vs {name_b}:")
    print(f"  Exact-pair overlap:      {len(exact_a & exact_b)}")
    print(f"  Korean-source overlap:   {len(ko_a & ko_b)}")
    print(f"  English-target overlap:  {len(en_a & en_b)}")

overlap_report(train_df, val_df, "train", "val")
overlap_report(train_df, test_df, "train", "test")
overlap_report(val_df, test_df, "val", "test")


## 11. Load NLLB Tokenizer

Loads the tokenizer for `facebook/nllb-200-distilled-600M`. NLLB uses
language-code tokens to control both segmentation context and generation
direction:
- Korean: `kor_Hang`
- English: `eng_Latn`

`src_lang` / `tgt_lang` are set per-call in the next sections (once for
Korean-as-source / English-as-target, and again reversed for the other
direction) rather than fixed once here, since this notebook needs both
directions.

In [ ]:
from transformers import AutoTokenizer

NLLB_MODEL_NAME = "facebook/nllb-200-distilled-600M"
KOREAN_LANG_CODE = "kor_Hang"
ENGLISH_LANG_CODE = "eng_Latn"

tokenizer = AutoTokenizer.from_pretrained(NLLB_MODEL_NAME)
print(f"Tokenizer class: {type(tokenizer).__name__}")
print(f"Vocab size:      {tokenizer.vocab_size}")

# Quick sanity check: confirm both language codes are valid tokens for this model.
for code_str in [KOREAN_LANG_CODE, ENGLISH_LANG_CODE]:
    token_id = tokenizer.convert_tokens_to_ids(code_str)
    print(f"Language code '{code_str}' -> token id {token_id} "
          f"({'OK' if token_id != tokenizer.unk_token_id else 'NOT FOUND -- check spelling'})")


In [ ]:
# --- Quick tokenization sanity check ------------------------------------------
sample_ko = df_clean["korean"].iloc[0]
sample_en = df_clean["english"].iloc[0]

tokenizer.src_lang = KOREAN_LANG_CODE
ko_encoded = tokenizer(sample_ko)
print(f"Korean sample:  {sample_ko}")
print(f"Token ids:      {ko_encoded['input_ids']}")
print(f"Tokens:         {tokenizer.convert_ids_to_tokens(ko_encoded['input_ids'])}")

print()
tokenizer.src_lang = ENGLISH_LANG_CODE
en_encoded = tokenizer(sample_en)
print(f"English sample: {sample_en}")
print(f"Token ids:      {en_encoded['input_ids']}")
print(f"Tokens:         {tokenizer.convert_ids_to_tokens(en_encoded['input_ids'])}")


## 12. Subword Token-Length Analysis

Character counts (Section 7) are not a reliable proxy for how long a sentence
is once tokenized into NLLB's subwords, especially for Korean. This section
tokenizes the **cleaned** training-split text with the real NLLB tokenizer, in
both roles (as a source sentence and as a target sentence) and both
languages, and reports the actual subword-token length distribution. Only the
train split is used here to avoid any influence from validation/test data on
this modeling decision.

`max_source_length` must cover both directions' source language (Korean
source for KO→EN, English source for EN→KO); `max_target_length` must
similarly cover both directions' target language.

In [ ]:
def token_lengths_as_source(texts, lang_code, batch_size=1000):
    tokenizer.src_lang = lang_code
    lengths = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tokenizer(batch)
        lengths.extend(len(ids) for ids in enc["input_ids"])
    return np.array(lengths)


def token_lengths_as_target(texts, lang_code, batch_size=1000):
    tokenizer.tgt_lang = lang_code
    lengths = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tokenizer(text_target=batch)
        lengths.extend(len(ids) for ids in enc["input_ids"])
    return np.array(lengths)


train_ko_texts = train_df["korean"].tolist()
train_en_texts = train_df["english"].tolist()

ko_src_token_lens = token_lengths_as_source(train_ko_texts, KOREAN_LANG_CODE)
en_src_token_lens = token_lengths_as_source(train_en_texts, ENGLISH_LANG_CODE)
ko_tgt_token_lens = token_lengths_as_target(train_ko_texts, KOREAN_LANG_CODE)
en_tgt_token_lens = token_lengths_as_target(train_en_texts, ENGLISH_LANG_CODE)

print("Token-length arrays computed for:")
print(f"  Korean as source:  {len(ko_src_token_lens)} sentences")
print(f"  English as source: {len(en_src_token_lens)} sentences")
print(f"  Korean as target:  {len(ko_tgt_token_lens)} sentences")
print(f"  English as target: {len(en_tgt_token_lens)} sentences")


In [ ]:
# --- Percentile table ----------------------------------------------------------
token_length_df = pd.DataFrame({
    "ko_as_source": pd.Series(ko_src_token_lens),
    "en_as_source": pd.Series(en_src_token_lens),
    "ko_as_target": pd.Series(ko_tgt_token_lens),
    "en_as_target": pd.Series(en_tgt_token_lens),
})

token_percentiles = token_length_df.describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99])
token_percentiles


In [ ]:
# --- Coverage / truncation-rate table at a few candidate max lengths ----------
candidate_lengths = [32, 48, 64, 96, 128, 160, 192, 256]

print("Truncation rate (% of sentences that would be cut) at candidate max lengths:\n")
print(f"{'max_len':>8} | {'ko_src':>8} | {'en_src':>8} | {'ko_tgt':>8} | {'en_tgt':>8}")
for L in candidate_lengths:
    row = [
        100 * (ko_src_token_lens > L).mean(),
        100 * (en_src_token_lens > L).mean(),
        100 * (ko_tgt_token_lens > L).mean(),
        100 * (en_tgt_token_lens > L).mean(),
    ]
    print(f"{L:>8} | {row[0]:>7.2f}% | {row[1]:>7.2f}% | {row[2]:>7.2f}% | {row[3]:>7.2f}%")


In [ ]:
# --- Length distribution plots (token level) -----------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(ko_src_token_lens, bins=40, alpha=0.6, label="Korean as source")
axes[0].hist(en_src_token_lens, bins=40, alpha=0.6, label="English as source")
axes[0].set_title("Subword token length -- source role")
axes[0].set_xlabel("Tokens")
axes[0].legend()

axes[1].hist(ko_tgt_token_lens, bins=40, alpha=0.6, label="Korean as target")
axes[1].hist(en_tgt_token_lens, bins=40, alpha=0.6, label="English as target")
axes[1].set_title("Subword token length -- target role")
axes[1].set_xlabel("Tokens")
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# --- Automated max_source_length / max_target_length recommendation -----------
# Recommendation rule: cover the p99 of whichever relevant distribution is
# larger, rounded UP to the nearest multiple of 8 (a minor efficiency
# convention for padded-batch matrix shapes on GPU), capped at 256 as a
# practical ceiling for a 600M-parameter model on a single T4 (14.6 GB) with
# a small batch size + gradient accumulation -- attention memory grows with
# sequence length, and this corpus's content (short-to-medium sentences, per
# the Section 7 character-length stats) does not need very long sequences.
def round_up_to_multiple(value, multiple=8):
    return int(np.ceil(value / multiple) * multiple)

P = 0.99
MAX_LEN_CAP = 256

source_p99 = max(np.percentile(ko_src_token_lens, P * 100),
                  np.percentile(en_src_token_lens, P * 100))
target_p99 = max(np.percentile(ko_tgt_token_lens, P * 100),
                  np.percentile(en_tgt_token_lens, P * 100))

recommended_max_source_length = min(round_up_to_multiple(source_p99), MAX_LEN_CAP)
recommended_max_target_length = min(round_up_to_multiple(target_p99), MAX_LEN_CAP)

print(f"p99 source token length (max over KO/EN as source): {source_p99:.1f}")
print(f"p99 target token length (max over KO/EN as target): {target_p99:.1f}")
print()
print(f"Recommended max_source_length: {recommended_max_source_length}")
print(f"Recommended max_target_length: {recommended_max_target_length}")
print()

trunc_src = 100 * (np.maximum(ko_src_token_lens, 0) > recommended_max_source_length).mean()
trunc_tgt = 100 * (np.maximum(ko_tgt_token_lens, 0) > recommended_max_target_length).mean()
print(f"Estimated truncation at recommended max_source_length "
      f"({recommended_max_source_length}): "
      f"KO {100 * (ko_src_token_lens > recommended_max_source_length).mean():.2f}%, "
      f"EN {100 * (en_src_token_lens > recommended_max_source_length).mean():.2f}%")
print(f"Estimated truncation at recommended max_target_length "
      f"({recommended_max_target_length}): "
      f"KO {100 * (ko_tgt_token_lens > recommended_max_target_length).mean():.2f}%, "
      f"EN {100 * (en_tgt_token_lens > recommended_max_target_length).mean():.2f}%")


**Review point:** the cell above prints the actual recommended
`max_source_length` / `max_target_length` for this dataset, plus the resulting
truncation rate. These are used starting in the pretrained-baseline cells
immediately below, and would carry through to fine-tuning in a later pass. If
the truncation rate at the recommended length looks too high (or the length
looks unnecessarily generous) once you see the real numbers, this is the
place to adjust the `P` (percentile) or `MAX_LEN_CAP` constants above and
re-run, before relying on these lengths further.

## 13. Pretrained NLLB Baseline — Before Fine-Tuning

This establishes the **zero-shot** performance of `facebook/nllb-200-distilled-600M`
on our held-out, leakage-safe **test split**, in both directions, before any
fine-tuning. This is the baseline that fine-tuning must beat to be considered
worthwhile at all (per the project's evidence-based acceptance rule).

Evaluated with:
- **SacreBLEU** (corpus BLEU, standard reproducible tokenization)
- **chrF** (via the `evaluate` library) — more robust than BLEU to Korean's
  morphology and flexible word order
- Qualitative examples (source / reference / prediction) for manual review

To keep this baseline pass fast, evaluation is capped at `MAX_EVAL_EXAMPLES`
per direction by default (randomly sampled from the test split with the fixed
seed). **This exact sampled subset is frozen and saved to disk** (`baseline_eval_df`
/ `baseline_eval_set.csv`) so the fine-tuned model can later be scored on
*precisely* the same source/reference pairs -- re-sampling with the same seed
after fine-tuning is not safe to rely on, since upstream DataFrame ordering or
content could differ by then; reusing the saved subset removes that risk.

In [ ]:
from transformers import AutoModelForSeq2SeqLM

print("Loading facebook/nllb-200-distilled-600M ... this may take a minute.")
model = AutoModelForSeq2SeqLM.from_pretrained(NLLB_MODEL_NAME)
model = model.to(device)

# Fix: NLLB's default generation_config sets a max_length that conflicts with
# passing max_new_tokens explicitly at generation time (as translate_batch()
# below does), producing a misleading warning and letting the wrong length
# control take effect. Clearing max_length here makes max_new_tokens the sole
# authority, consistently, for every generate() call using this model object
# -- during this baseline pass AND later during/after fine-tuning, since it's
# the same in-memory model instance throughout this notebook.
model.generation_config.max_length = None

model.eval()
print(f"Model loaded on: {device}")


In [ ]:
MAX_EVAL_EXAMPLES = 500   # cap per direction for this baseline pass; raise later if needed
EVAL_BATCH_SIZE = 16
NUM_BEAMS = 4

set_seed(SEED)

# Freeze the exact evaluation subset now, so it can be reused unchanged when
# scoring the fine-tuned model later. Do NOT re-sample test_df again after
# fine-tuning, even with the same SEED -- upstream DataFrame ordering/content
# could differ between runs/sessions, which would silently break the
# apples-to-apples comparison. `baseline_eval_indices` records positions
# within `test_df` as it exists right now; `baseline_eval_df` is the actual
# frozen data, and is additionally saved to disk for reuse across sessions.
if len(test_df) <= MAX_EVAL_EXAMPLES:
    baseline_eval_indices = test_df.index.tolist()
else:
    baseline_eval_indices = test_df.sample(
        n=MAX_EVAL_EXAMPLES, random_state=SEED).index.tolist()

baseline_eval_df = test_df.loc[baseline_eval_indices].reset_index(drop=True)

BASELINE_EVAL_PATH = pathlib.Path("baseline_eval_set.csv")
baseline_eval_df.to_csv(BASELINE_EVAL_PATH, index=False)

print(f"Frozen baseline evaluation subset: {len(baseline_eval_df)} of {len(test_df)} test examples.")
print(f"Saved to: {BASELINE_EVAL_PATH.resolve()}")
print("Reuse this exact file/DataFrame (not a fresh sample) when evaluating "
      "the fine-tuned model, for a fair pretrained-vs-fine-tuned comparison.")


@torch.no_grad()
def translate_batch(texts, src_lang, tgt_lang, max_source_length, max_target_length,
                     batch_size=EVAL_BATCH_SIZE, num_beams=NUM_BEAMS):
    tokenizer.src_lang = src_lang
    forced_bos_token_id = tokenizer.convert_tokens_to_ids(tgt_lang)
    outputs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True,
                            truncation=True, max_length=max_source_length).to(device)
        generated = model.generate(
            **inputs,
            forced_bos_token_id=forced_bos_token_id,
            max_new_tokens=max_target_length,
            num_beams=num_beams,
        )
        decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
        outputs.extend(decoded)
    return outputs


In [ ]:
# --- Korean -> English -----------------------------------------------------
ko_en_sources = baseline_eval_df["korean"].tolist()
ko_en_references = baseline_eval_df["english"].tolist()

ko_en_predictions = translate_batch(
    ko_en_sources, src_lang=KOREAN_LANG_CODE, tgt_lang=ENGLISH_LANG_CODE,
    max_source_length=recommended_max_source_length,
    max_target_length=recommended_max_target_length,
)
print(f"Generated {len(ko_en_predictions)} KO->EN translations.")


In [ ]:
# --- English -> Korean -----------------------------------------------------
en_ko_sources = baseline_eval_df["english"].tolist()
en_ko_references = baseline_eval_df["korean"].tolist()

en_ko_predictions = translate_batch(
    en_ko_sources, src_lang=ENGLISH_LANG_CODE, tgt_lang=KOREAN_LANG_CODE,
    max_source_length=recommended_max_source_length,
    max_target_length=recommended_max_target_length,
)
print(f"Generated {len(en_ko_predictions)} EN->KO translations.")


In [ ]:
# --- BLEU + chrF for both directions -------------------------------------------
import sacrebleu

chrf_metric = evaluate.load("chrf")

def score_direction(predictions, references, direction_name):
    bleu = sacrebleu.corpus_bleu(predictions, [references])
    chrf = chrf_metric.compute(predictions=predictions, references=[[r] for r in references])
    print(f"[{direction_name}] BLEU: {bleu.score:.2f}   chrF: {chrf['score']:.2f}")
    return bleu.score, chrf["score"]

print("Pretrained NLLB baseline (before fine-tuning):\n")
ko_en_bleu, ko_en_chrf = score_direction(ko_en_predictions, ko_en_references, "KO -> EN")
en_ko_bleu, en_ko_chrf = score_direction(en_ko_predictions, en_ko_references, "EN -> KO")

pretrained_baseline_results = {
    "ko_en": {"bleu": ko_en_bleu, "chrf": ko_en_chrf},
    "en_ko": {"bleu": en_ko_bleu, "chrf": en_ko_chrf},
    "n_eval_examples": len(baseline_eval_df),
}
print(f"\n{pretrained_baseline_results}")


In [ ]:
# --- Qualitative examples -------------------------------------------------------
set_seed(SEED)
qual_idx = np.random.choice(len(baseline_eval_df), size=min(6, len(baseline_eval_df)), replace=False)

print("KO -> EN qualitative examples:\n")
for i in qual_idx:
    print(f"Source (KO):    {ko_en_sources[i]}")
    print(f"Reference (EN): {ko_en_references[i]}")
    print(f"Prediction (EN):{ko_en_predictions[i]}")
    print("-" * 60)

print("\nEN -> KO qualitative examples:\n")
for i in qual_idx:
    print(f"Source (EN):    {en_ko_sources[i]}")
    print(f"Reference (KO): {en_ko_references[i]}")
    print(f"Prediction (KO):{en_ko_predictions[i]}")
    print("-" * 60)


### Section 13 results (recorded, per your review)

- **Pretrained baseline (frozen reference):** KO→EN BLEU 15.97 / chrF 33.81;
  EN→KO BLEU 3.88 / chrF 17.08, on the 500-example frozen `baseline_eval_df`.
- EN→KO is markedly weaker than KO→EN, with qualitative omissions/meaning
  loss — consistent with NLLB's English-centric pretraining data generally
  being stronger going *into* English than *out of* it for a mid-resource
  pair like Korean.

Sections 14 onward proceed to fine-tuning setup, per your approval. Nothing
in this section is fine-tuned yet — the actual `trainer.train()` call is
intentionally withheld until you approve the configuration below.

## 14. Preserve Pretrained Baseline Results

Before touching anything related to fine-tuning, the pretrained baseline
results computed in Section 13 (`pretrained_baseline_results`) are persisted
to disk now, under their own name, completely separate from anything
fine-tuning will produce. Fine-tuned results will be stored in a
**different** variable (`finetuned_eval_results`, added in a later section)
and a different file — `pretrained_baseline_results` / `pretrained_baseline_results.json`
are never reassigned or overwritten from this point on.

In [ ]:
import json as _json

assert "pretrained_baseline_results" in dir(), (
    "pretrained_baseline_results is missing -- re-run Section 13 before continuing."
)

PRETRAINED_BASELINE_PATH = pathlib.Path("pretrained_baseline_results.json")
with open(PRETRAINED_BASELINE_PATH, "w", encoding="utf-8") as f:
    _json.dump(pretrained_baseline_results, f, indent=2)

print("Preserved pretrained baseline results (will not be modified further):")
print(pretrained_baseline_results)
print(f"\nSaved to: {PRETRAINED_BASELINE_PATH.resolve()}")


## 15. Bidirectional Training Dataset Construction

Built **only** from `train_df` (fine-tuning data) and `val_df` (model
selection / early stopping) — `test_df` and `baseline_eval_df` are never
touched here, per the requirement that the test set stays untouched until
final evaluation.

**Method:** rather than tokenizing one mixed-direction dataset with a
tokenizer whose `src_lang`/`tgt_lang` state would need to vary row-by-row
(fragile to get right, and awkward to batch), each direction is built and
tokenized as its own dataset — exactly the same pattern already used and
validated in Section 12's token-length analysis — then the two directions
are concatenated and shuffled together. This keeps every tokenization call
in a single, auditable, fixed `src_lang`/`tgt_lang` state.

`max_source_length = 64` and `max_target_length = 64` (fixed, as specified)
are applied identically to every direction.

In [ ]:
from datasets import Dataset, concatenate_datasets

MAX_SOURCE_LENGTH = 64
MAX_TARGET_LENGTH = 64


def tokenize_direction_batch(batch, src_lang, tgt_lang):
    tokenizer.src_lang = src_lang
    tokenizer.tgt_lang = tgt_lang
    model_inputs = tokenizer(
        batch["source"], max_length=MAX_SOURCE_LENGTH, truncation=True)
    labels = tokenizer(
        text_target=batch["target"], max_length=MAX_TARGET_LENGTH, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


def build_direction_dataset(df_split, source_col, target_col, src_lang, tgt_lang,
                             map_batch_size=1000):
    """Build and tokenize one translation direction from one split."""
    raw = Dataset.from_dict({
        "source": df_split[source_col].tolist(),
        "target": df_split[target_col].tolist(),
    })
    tokenized = raw.map(
        lambda batch: tokenize_direction_batch(batch, src_lang, tgt_lang),
        batched=True,
        batch_size=map_batch_size,
        remove_columns=["source", "target"],
    )
    return tokenized


In [ ]:
# --- Build each direction, from TRAIN and VALIDATION splits only --------------
train_ko2en = build_direction_dataset(train_df, "korean", "english", KOREAN_LANG_CODE, ENGLISH_LANG_CODE)
train_en2ko = build_direction_dataset(train_df, "english", "korean", ENGLISH_LANG_CODE, KOREAN_LANG_CODE)

val_ko2en = build_direction_dataset(val_df, "korean", "english", KOREAN_LANG_CODE, ENGLISH_LANG_CODE)
val_en2ko = build_direction_dataset(val_df, "english", "korean", ENGLISH_LANG_CODE, KOREAN_LANG_CODE)

# --- Concatenate directions and shuffle (fixed seed) ---------------------------
train_dataset = concatenate_datasets([train_ko2en, train_en2ko]).shuffle(seed=SEED)
val_dataset = concatenate_datasets([val_ko2en, val_en2ko]).shuffle(seed=SEED)

print("Bidirectional dataset construction (auditable counts):\n")
print(f"  train_df rows (source):        {len(train_df)}")
print(f"  train KO->EN examples:         {len(train_ko2en)}")
print(f"  train EN->KO examples:         {len(train_en2ko)}")
print(f"  train_dataset (combined):      {len(train_dataset)}  "
      f"(expected: {len(train_ko2en)} + {len(train_en2ko)} = {len(train_ko2en) + len(train_en2ko)})")
print()
print(f"  val_df rows (source):          {len(val_df)}")
print(f"  val KO->EN examples:           {len(val_ko2en)}")
print(f"  val EN->KO examples:           {len(val_en2ko)}")
print(f"  val_dataset (combined):        {len(val_dataset)}  "
      f"(expected: {len(val_ko2en)} + {len(val_en2ko)} = {len(val_ko2en) + len(val_en2ko)})")

assert len(train_dataset) == len(train_ko2en) + len(train_en2ko)
assert len(val_dataset) == len(val_ko2en) + len(val_en2ko)


## 16. Data Collator

`DataCollatorForSeq2Seq` dynamically pads each batch (source input_ids,
attention_mask, and labels) to the longest example in that batch, and sets
padded label positions to `-100` so they are excluded from the loss.

In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
)
print("Data collator ready.")


## 17. Training Arguments (Conservative, Catastrophic-Forgetting-Averse)

| Setting | Value | Why |
|---|---|---|
| `learning_rate` | `2e-5` | Standard conservative rate for continuing pretraining/fine-tuning a large pretrained seq2seq Transformer; low enough to adapt to our data without overwriting NLLB's existing multilingual knowledge |
| `num_train_epochs` | `3` (max) | Small dataset relative to pretraining scale — more epochs risks overfitting/forgetting; early stopping (below) can stop sooner |
| `per_device_train_batch_size` | `4` | Reduced from an initial `8` as a memory-safety margin: NLLB-200's ~256K-token vocabulary means decoder logits alone are large (batch × seq_len × 256K), on top of model weights, optimizer state, gradients, and activations — `4` lowers peak VRAM without changing the effective batch size |
| `gradient_accumulation_steps` | `8` | Increased from `4` to compensate, keeping the effective batch size identical to the original plan |
| **Effective batch size** | `4 \u00d7 8 \u00d7 1 GPU = 32` | Unchanged from the original plan — computed, not just declared — printed below from these exact values |
| `fp16` | `True` | Required on T4 for both memory savings and speed |
| `gradient_checkpointing` | `True` | Extra memory safety margin: a 600M-parameter model's AdamW optimizer state (momentum + variance, both fp32) plus fp32 master weights alone approach ~10–12 GB before any activations, on a 14.6 GB card — checkpointing trades some compute for significantly less activation memory |
| `eval_strategy` / `save_strategy` | `"epoch"` / `"epoch"` | Matches your requirement to evaluate/checkpoint each epoch; both must match for `load_best_model_at_end` to work |
| `load_best_model_at_end` | `True` | Required so the final model is the best-validation checkpoint, not just the last one |
| `metric_for_best_model` | `"eval_loss"` (minimize) | Fast, computed every eval step for free (no extra generation calls during training); a real BLEU/chrF comparison happens once, at the end, on the untouched `baseline_eval_df` — not used to pick the checkpoint, per your requirement not to use the test set for tuning |
| `save_total_limit` | `2` | Keeps disk usage bounded |
| `warmup_ratio` | `0.1` | Standard practice to avoid a large learning-rate shock at the very start of fine-tuning a pretrained model |
| `weight_decay` | `0.01` | Standard, mild regularization |
| `predict_with_generate` | `False` | Generation-based metrics are computed once, separately, after training — not every eval step, to keep training fast |
| `report_to` | `"none"` | Avoids a Colab prompt for an unused experiment-tracking service |
| `seed` | `SEED` (42) | Reproducibility, consistent with the rest of this notebook |

**Early stopping:** `EarlyStoppingCallback(early_stopping_patience=1)` — with
only up to 3 epochs, a patience of 1 means training stops as soon as one
epoch fails to improve `eval_loss` over the best seen so far, which is
exactly the kind of early, cheap catastrophic-forgetting/overfitting signal
this conservative strategy is meant to catch.

In [ ]:
from transformers import Seq2SeqTrainingArguments

LEARNING_RATE = 2e-5
NUM_TRAIN_EPOCHS = 3
PER_DEVICE_TRAIN_BATCH_SIZE = 4
PER_DEVICE_EVAL_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 8
EARLY_STOPPING_PATIENCE = 1

n_gpus = max(torch.cuda.device_count(), 1)
effective_batch_size = PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS * n_gpus

steps_per_epoch = int(np.ceil(len(train_dataset) / effective_batch_size))
estimated_total_steps = steps_per_epoch * NUM_TRAIN_EPOCHS

print(f"GPUs detected:                {n_gpus}")
print(f"Effective batch size:         {PER_DEVICE_TRAIN_BATCH_SIZE} x "
      f"{GRADIENT_ACCUMULATION_STEPS} x {n_gpus} = {effective_batch_size}")
print(f"Training examples:            {len(train_dataset)}")
print(f"Estimated steps per epoch:    {steps_per_epoch}")
print(f"Estimated total steps (max, {NUM_TRAIN_EPOCHS} epochs): {estimated_total_steps}")
print("(Actual total steps may be lower if early stopping triggers before "
      f"{NUM_TRAIN_EPOCHS} epochs complete.)")

training_args = Seq2SeqTrainingArguments(
    output_dir="nllb_finetuned_checkpoints",
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=True,
    gradient_checkpointing=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    logging_strategy="steps",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    predict_with_generate=False,
    report_to="none",
    seed=SEED,
)
print("\nSeq2SeqTrainingArguments constructed (not yet used to train).")


## 18. Trainer Setup

Instantiates `Seq2SeqTrainer` with the components above. `model.config.use_cache`
is explicitly disabled, since it must be off during training whenever
`gradient_checkpointing=True` (the two are incompatible while training,
independent of this specific model) — it will be re-enabled after training,
for efficient generation at evaluation/inference time.

**No training happens in this cell.** Instantiating `Seq2SeqTrainer` does not
run anything by itself; `trainer.train()` is a separate, deliberately
withheld call.

In [ ]:
from transformers import Seq2SeqTrainer, EarlyStoppingCallback

model.config.use_cache = False  # required alongside gradient_checkpointing during training

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)

print("Seq2SeqTrainer constructed.")
print(f"Train dataset:      {len(train_dataset)} examples")
print(f"Validation dataset: {len(val_dataset)} examples")
print(f"Test set / baseline_eval_df: untouched, not referenced anywhere above.")


### Approved configuration — proceeding to training

The configuration above (`per_device_train_batch_size=4`,
`gradient_accumulation_steps=8`, effective batch size `32`) is approved.
Sections 19 onward add GPU memory preparation, the actual `trainer.train()`
call (guarded against silently mishandling an out-of-memory error), and the
full post-training evaluation against the frozen `baseline_eval_df`.

## 19. GPU Memory Preparation

Frees any memory held by earlier cells (e.g. the pretrained-baseline
generation calls in Section 13) before training starts, and reports current
allocated/reserved GPU memory as a reference point.

In [ ]:
import gc


def print_gpu_memory(label=""):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / (1024 ** 3)
        reserved = torch.cuda.memory_reserved() / (1024 ** 3)
        print(f"{label} GPU memory -- allocated: {allocated:.2f} GB, reserved: {reserved:.2f} GB")
    else:
        print(f"{label} No CUDA device available.")


gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print_gpu_memory("Before training:")


## 20. Training Execution

Runs `trainer.train()`. If a CUDA out-of-memory error occurs, it is caught
**only long enough to print a clear, actionable message** — hyperparameters
are not changed and training is not silently retried. The exception is then
re-raised, which stops the notebook at this cell.

**If this happens:** manually change `PER_DEVICE_TRAIN_BATCH_SIZE` from `4`
to `2` and `GRADIENT_ACCUMULATION_STEPS` from `8` to `16` in Section 17
(effective batch size stays `32`), re-run Section 17 (Training Arguments) and
Section 18 (Trainer Setup) to rebuild `training_args`/`trainer` with the new
values, then re-run this cell.

In [ ]:
import time

train_start_time = time.time()

try:
    train_result = trainer.train()
except RuntimeError as e:
    if "out of memory" in str(e).lower():
        print("=" * 70)
        print("CUDA OUT OF MEMORY during trainer.train().")
        print("Training has stopped. No hyperparameters were changed automatically.")
        print()
        print("Manual next step: in Section 17, set")
        print("  PER_DEVICE_TRAIN_BATCH_SIZE = 2")
        print("  GRADIENT_ACCUMULATION_STEPS = 16")
        print("(effective batch size stays 4x8=32 -> 2x16=32), then re-run Section 17,")
        print("Section 18, and this cell.")
        print("=" * 70)
    raise

train_end_time = time.time()
training_time_seconds = train_end_time - train_start_time

print(f"\nTraining completed in {training_time_seconds / 60:.1f} minutes "
      f"({training_time_seconds:.1f} seconds).")
print_gpu_memory("After training:")


## 21. Post-Training Report

Saves the full train/validation loss history to disk and reports which
checkpoint was selected as best (by `eval_loss`), its value, and total
training time.

In [ ]:
log_history_df = pd.DataFrame(trainer.state.log_history)

LOG_HISTORY_PATH = pathlib.Path("finetune_log_history.csv")
log_history_df.to_csv(LOG_HISTORY_PATH, index=False)
print(f"Saved training/validation log history to: {LOG_HISTORY_PATH.resolve()}")

best_checkpoint = trainer.state.best_model_checkpoint
best_metric = trainer.state.best_metric

print(f"\nBest checkpoint:            {best_checkpoint}")
print(f"Best validation eval_loss:  {best_metric}")
print(f"Total training time:        {training_time_seconds / 60:.1f} minutes")


In [ ]:
# --- Loss curves ---------------------------------------------------------------
train_logs = log_history_df[log_history_df.get("loss").notna()] if "loss" in log_history_df else pd.DataFrame()
eval_logs = log_history_df[log_history_df.get("eval_loss").notna()] if "eval_loss" in log_history_df else pd.DataFrame()

fig, ax = plt.subplots(figsize=(7, 4))
if not train_logs.empty:
    ax.plot(train_logs["step"], train_logs["loss"], label="train_loss")
if not eval_logs.empty:
    ax.plot(eval_logs["step"], eval_logs["eval_loss"], label="eval_loss", marker="o")
ax.set_xlabel("step")
ax.set_ylabel("loss")
ax.set_title("Fine-tuning loss curves")
ax.legend()
plt.show()


## 22. Restore Inference Settings and Save the Fine-Tuned Model

`use_cache` is re-enabled now that training (which required it disabled,
alongside gradient checkpointing) is finished. The
`generation_config.max_length = None` fix from Section 13 is preserved and
re-asserted here for safety. The fine-tuned model and tokenizer are saved to
their **own** directory, separate from anything related to Version 1 or the
pretrained-baseline model state.

In [ ]:
model.config.use_cache = True
model.generation_config.max_length = None  # preserved fix -- re-asserted, not reintroduced differently

print(f"model.config.use_cache:              {model.config.use_cache}")
print(f"model.generation_config.max_length:  {model.generation_config.max_length}")

FINETUNED_MODEL_DIR = pathlib.Path("nllb_finetuned_ko_en_bidirectional")
trainer.save_model(str(FINETUNED_MODEL_DIR))
tokenizer.save_pretrained(str(FINETUNED_MODEL_DIR))
print(f"\nFine-tuned model + tokenizer saved to: {FINETUNED_MODEL_DIR.resolve()}")


## 23. Evaluate the Fine-Tuned Model — Same Frozen Evaluation Set

Reuses, unchanged: `baseline_eval_df`, the `translate_batch()` function, the
`recommended_max_source_length` / `recommended_max_target_length` values, and
`NUM_BEAMS` / `EVAL_BATCH_SIZE` — all from Section 13 — together with
`ko_en_sources` / `ko_en_references` / `en_ko_sources` / `en_ko_references`,
which were derived directly from `baseline_eval_df` and never resampled.
Only the model has changed (now fine-tuned); every other input to this
evaluation is identical to the pretrained-baseline pass, which is what makes
the comparison in Section 24 fair.

In [ ]:
model.eval()

print("Translating the frozen baseline_eval_df with the FINE-TUNED model "
      "(same sources, same generation settings as the pretrained baseline)...")

ko_en_predictions_ft = translate_batch(
    ko_en_sources, src_lang=KOREAN_LANG_CODE, tgt_lang=ENGLISH_LANG_CODE,
    max_source_length=recommended_max_source_length,
    max_target_length=recommended_max_target_length,
)
print(f"Generated {len(ko_en_predictions_ft)} KO->EN translations (fine-tuned).")

en_ko_predictions_ft = translate_batch(
    en_ko_sources, src_lang=ENGLISH_LANG_CODE, tgt_lang=KOREAN_LANG_CODE,
    max_source_length=recommended_max_source_length,
    max_target_length=recommended_max_target_length,
)
print(f"Generated {len(en_ko_predictions_ft)} EN->KO translations (fine-tuned).")


In [ ]:
# --- BLEU + chrF for the fine-tuned model (stored SEPARATELY from the
# pretrained baseline -- pretrained_baseline_results is not touched) ----------
print("Fine-tuned NLLB (after bidirectional fine-tuning):\n")
ko_en_bleu_ft, ko_en_chrf_ft = score_direction(ko_en_predictions_ft, ko_en_references, "KO -> EN (fine-tuned)")
en_ko_bleu_ft, en_ko_chrf_ft = score_direction(en_ko_predictions_ft, en_ko_references, "EN -> KO (fine-tuned)")

finetuned_eval_results = {
    "ko_en": {"bleu": ko_en_bleu_ft, "chrf": ko_en_chrf_ft},
    "en_ko": {"bleu": en_ko_bleu_ft, "chrf": en_ko_chrf_ft},
    "n_eval_examples": len(baseline_eval_df),
}

FINETUNED_RESULTS_PATH = pathlib.Path("finetuned_eval_results.json")
with open(FINETUNED_RESULTS_PATH, "w", encoding="utf-8") as f:
    _json.dump(finetuned_eval_results, f, indent=2)

print(f"\nSaved fine-tuned evaluation results to: {FINETUNED_RESULTS_PATH.resolve()}")
print(f"\npretrained_baseline_results (unchanged, preserved from Section 14): {pretrained_baseline_results}")


## 24. Pretrained vs Fine-Tuned Comparison Table

In [ ]:
comparison_rows = []
for direction_key, direction_label in [("ko_en", "KO -> EN"), ("en_ko", "EN -> KO")]:
    pre = pretrained_baseline_results[direction_key]
    fte = finetuned_eval_results[direction_key]
    comparison_rows.append({
        "direction": direction_label,
        "pretrained_BLEU": pre["bleu"],
        "finetuned_BLEU": fte["bleu"],
        "delta_BLEU": fte["bleu"] - pre["bleu"],
        "pretrained_chrF": pre["chrf"],
        "finetuned_chrF": fte["chrf"],
        "delta_chrF": fte["chrf"] - pre["chrf"],
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df


## 25. Qualitative Comparison — Same Fixed Examples

Uses the exact same `qual_idx` rows selected in Section 13, so pretrained and
fine-tuned outputs are compared on identical sentences, not a fresh sample.

In [ ]:
print("KO -> EN qualitative comparison (same examples as the pretrained baseline):\n")
for i in qual_idx:
    print(f"Source (KO):      {ko_en_sources[i]}")
    print(f"Reference (EN):   {ko_en_references[i]}")
    print(f"Pretrained (EN):  {ko_en_predictions[i]}")
    print(f"Fine-tuned (EN):  {ko_en_predictions_ft[i]}")
    print("-" * 60)

print("\nEN -> KO qualitative comparison (same examples as the pretrained baseline):\n")
for i in qual_idx:
    print(f"Source (EN):      {en_ko_sources[i]}")
    print(f"Reference (KO):   {en_ko_references[i]}")
    print(f"Pretrained (KO):  {en_ko_predictions[i]}")
    print(f"Fine-tuned (KO):  {en_ko_predictions_ft[i]}")
    print("-" * 60)


## 26. Per-Direction Verdict and Acceptance Decision

Per the project's evidence-based acceptance rule: fine-tuning is not accepted
just because training completed. Each direction is explicitly labeled
improved / degraded / roughly unchanged (using a ±1.0 BLEU-point band as
"roughly unchanged" -- adjust this threshold if you want a stricter or looser
call). If the combined effect is negative, this cell prints a diagnostic
checklist rather than a recommendation to ship the checkpoint anyway.

In [ ]:
def verdict(delta_bleu, threshold=1.0):
    if delta_bleu > threshold:
        return "IMPROVED"
    if delta_bleu < -threshold:
        return "DEGRADED"
    return "ROUGHLY UNCHANGED"


print("Per-direction verdict (+/-1.0 BLEU point treated as \'roughly unchanged\'):\n")
for row in comparison_rows:
    v = verdict(row["delta_BLEU"])
    print(f"{row['direction']}: {v}")
    print(f"  BLEU: {row['pretrained_BLEU']:.2f} -> {row['finetuned_BLEU']:.2f}  (delta {row['delta_BLEU']:+.2f})")
    print(f"  chrF: {row['pretrained_chrF']:.2f} -> {row['finetuned_chrF']:.2f}  (delta {row['delta_chrF']:+.2f})")
    print()

overall_bleu_delta = sum(row["delta_BLEU"] for row in comparison_rows)

if overall_bleu_delta < 0:
    print("=" * 70)
    print("Combined BLEU delta across both directions is NEGATIVE.")
    print("Per the project\'s acceptance rule, this checkpoint is NOT automatically")
    print("accepted as the production model. Diagnostic checklist:")
    print("  1. Learning rate too high for this dataset size -> retry with e.g. 1e-5")
    print("  2. Overfitting -> check the loss curve above: did eval_loss rise while")
    print("     train_loss kept falling?")
    print("  3. Noisy/narrow training data -> revisit Section 8 cleaning and Section 9")
    print("     deduplication decisions")
    print("  4. Catastrophic forgetting -> if degradation is broad/severe across both")
    print("     directions, consider fewer epochs, a lower learning rate, or a smaller")
    print("     number of fine-tuning steps")
    print("=" * 70)
else:
    print("Combined BLEU delta across both directions is non-negative. Review the "
          "per-direction verdicts above individually, though -- one direction can "
          "still degrade even when the combined delta is positive, and per-direction "
          "acceptance should be judged on its own merits, not just the combined figure.")


In [ ]:
from huggingface_hub import notebook_login
notebook_login()

HF_REPO_ID = "ayashaaban/nllb-ko-en-finetuned"

model.push_to_hub(HF_REPO_ID)
tokenizer.push_to_hub(HF_REPO_ID)
print("Done:", f"https://huggingface.co/{HF_REPO_ID}")